# ---------- Final Manual - Please Don't Hate Me If This Ends Bad... ----------

### Last year's returns: 

'Refrigerators': 0.020713448079427082, (Social Hype)

'Earrings': 0.12367402886284722, (Merger with competitor)

'Blankets': -0.32888405330882353, (Regulatry Banning of product)

'Sleds': -0.2829537109375, (TAXES tripled)

'Sculptures': 0.19637428385416666, (natural disaster but plans to recoup)

'PS6': 0.3095545703125, (Games surging)

'Serum': -0.8157531666666666, (product scam)

'Lamps': 6.103515625e-05, (no fucking idea)

'Chocolate': -0.000404595947265625}, (court case but weird and not credible)


### Equivalent this year: 




Statues - Red flag  

E blankets - Needles 

PS6 - VR

Ranch - Earrings (Merger)

Fridges - Haystacks 

Dog Sled - Solar Panels

Refrigerators -> Online Hype 


# ---------- Analysis... ----------

In [21]:
import numpy as np
import cvxpy as cp
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wl, wlexpr

In [ ]:

sentiments = {
    'Haystacks': '+',
    'Ranch': '++',
    'Needles': '----',
    'Solar Panels': '---',
    'Red Flags': '++',
    'VR Monocle': '+++',
    'Coffee': '-----',
    'Moonshine': '-',
    'Shirts': '+',
}
returns = {
    '+': 0.02,
    '++': 0.15,
    '+++': 0.30,
    '-': -0.02,
    '--': -0.15,
    '---': -0.3,
    '----': -0.5
}

products = list(sentiments.keys())

In [55]:
rets = np.array([returns[sentiments[products[i-1]]] for i in range(1,10)])
pi = cp.Variable(9)
objective = cp.Minimize(120 * cp.sum_squares(pi) - 10000 * rets.T @ pi)
constraints = [cp.norm(pi, 1) <= 100]
prob = cp.Problem(objective, constraints)

prob.solve()
print('Optimal allocation without integer constraints:')
for i in range(9):
    print("Position in ", products[i], ': ', f"{pi.value[i]:,.2f}", '%', sep='')

Polishing not needed - no active set detected at optimal point
Optimal allocation without integer constraints:
Position in Haystacks: 0.83%
Position in Ranch: 6.25%
Position in Needles: -16.67%
Position in Solar Panels: -12.50%
Position in Red Flags: 6.25%
Position in VR Monocle: 12.50%
Position in Coffee: -12.50%
Position in Moonshine: -0.83%
Position in Shirts: 0.83%


In [56]:

s1 = ' + '.join(['('+str(returns[sentiments[products[i-1]]])+')*p'+str(i)+'*10000-120*(p'+str(i)+')^2' for i in range(1,10)])
s2 = ' + '.join(['Abs[p'+str(i)+']' for i in range(1,10)]) + '<=100,'
s3 = ', '.join(['Element[p'+str(i)+', Integers]' for i in range(1,10)])
s4 = ', '.join(['p'+str(i) for i in range(1,10)])

In [57]:
with  WolframLanguageSession() as session:
    session.start()
    val_max, sol = session.evaluate(wlexpr('NMaximize[{'+s1+','+s2+s3+'}, {'+s4+'}]'))

In [58]:

print("Maximum profit achievable:", val_max)
print("Percentage of capital used: ", sum([abs(el[1]) for el in sol]), '%', sep='')

Maximum profit achievable: 99080.0
Percentage of capital used: 69%


In [59]:
for i in range(9):
    print("Position in ", products[i], ': ', sol[i][1], '%', sep='')

Position in Haystacks: 1%
Position in Ranch: 6%
Position in Needles: -17%
Position in Solar Panels: -12%
Position in Red Flags: 6%
Position in VR Monocle: 12%
Position in Coffee: -13%
Position in Moonshine: -1%
Position in Shirts: 1%


In [60]:
# This code is heavily inspired by Round5.ipynb in gabsens's IMC-Prosperity-2-Manual GitHub repository
# Link: https://github.com/gabsens/IMC-Prosperity-2-Manual/blob/master/Round5.ipynb

# https://www.google.com/search?q=numpy+t+%40&rlz=1C1VDKB_enUS970US970&oq=numpy+t+%40&gs_lcrp=EgZjaHJvbWUyBggAEEUYOdIBCDE2ODNqMGo3qAIAsAIA&sourceid=chrome&ie=UTF-8
# import numpy as np

def make_product_multipliers(sentiments, sentiment_multipliers):
    d = {}
    for product, sentiment in sentiments.items():
        d[product] = sentiment_multipliers[sentiment]

    return d

def print_format_one(products, optimal_profits, initial_capital):
    total_expected_profit = 0
    total_percent = 0

    print("------- OPTIMAL %'S TO ALLOCATE PER PRODUCT (POSITIVE = BUY, NEGATIVE = SELL) -------\n")
    for product in products:
        optimal_pi_i = round(optimal_profits[product][0], 2)
        expected_profit = round(optimal_profits[product][1], 2)

        print(f"{product}: {optimal_pi_i}%")
        print(f"    Expected profit: {expected_profit}\n")

        total_percent += abs(optimal_pi_i)
        total_expected_profit += expected_profit
    
    print(f"Total expected profit: {round(total_expected_profit, 2)}")
    print(f"Total %/capital used: {total_percent}% = {initial_capital * (total_percent / 100)} seashells")

def print_format_two(products, optimal_profits, initial_capital):
    total_expected_profit = 0
    total_percent = 0

    print("------- OPTIMAL %'S TO ALLOCATE PER PRODUCT -------\n")
    for product in products:
        optimal_pi_i = round(optimal_profits[product][0], 2)
        expected_profit = round(optimal_profits[product][1], 2)

        if optimal_pi_i > 0:
            print(f"{product}: {optimal_pi_i}% BUY")
        else:
            print(f"{product}: {-1 * optimal_pi_i}% SELL")

        print(f"    Expected profit: {expected_profit}\n")

        total_percent += abs(optimal_pi_i)
        total_expected_profit += expected_profit
    
    print(f"Total expected profit: {round(total_expected_profit, 2)}")
    print(f"Total %/capital used: {total_percent}% = {initial_capital * (total_percent / 100)} seashells")

""" sentiments = {
    'Haystacks': '+',
    'Ranch sauce': '++',
    'Cacti Needle': '----',
    'Solar panels': '--',
    'Red Flags': '+',
    'VR Monocle': '+++',
    'Quantum Coffee': '---',
    'Moonshine': '-',
    'Striped shirts': '+'
}

sentiment_multipliers = {
    '+': 0.05,
    '++': 0.15,
    '+++': 0.25,
    '-': -0.05,
    '--': -0.11,
    '---': -0.43,
    '----': -0.6
} """

sentiments = {
    'Refrigerators': '+',
    'Earrings': '++',
    'Blankets': '---',
    'Sleds': '--',
    'Sculptures': '++',
    'PS6': '+++',
    'Serum': '----',
    'Lamps': '+',
    'Chocolate': '-'
}

sentiment_multipliers = {
    '+': 0.05,
    '++': 0.15,
    '+++': 0.25,
    '-': -0.05,
    '--': -0.1,
    '---': -0.4,
    '----': -0.6
}

products = list(sentiments.keys())
product_multipliers = make_product_multipliers(sentiments, sentiment_multipliers)

#print(products)
#print(product_multipliers)

# pi_i = % allocated (optimize this)
# r_i = anticipated return multiplier I guess? (product_multipliers)
# Fee = 120 * pi_i^2
initial_capital = 1_000_000

# profit_dictionary = {product: [pi_i, associated profit]}
optimal_profits = {}

for product in products:
    r_i = product_multipliers[product]
    pi_i = -100

    while True:
        # Stop at 100% (include 100% in the calculations though)
        if pi_i > 100:
            break
        
        # In the GitHub: profit = 7500 * r_i * pi_i - 90 * pi_i^2
        profit = (initial_capital * r_i * (pi_i / 100)) - (120 * pi_i * pi_i)

        if optimal_profits.get(product) is None:
            optimal_profits[product] = [pi_i, profit]
        
        else:
            if profit > optimal_profits[product][1]:
                optimal_profits[product] = [pi_i, profit]

        # Test all percentages from -100% to 100% with up to 2 decimal places 
        pi_i += 0.01

#print_format_one(products, optimal_profits, initial_capital)
print_format_two(products, optimal_profits, initial_capital)

------- OPTIMAL %'S TO ALLOCATE PER PRODUCT -------

Refrigerators: 2.08% BUY
    Expected profit: 520.83

Earrings: 6.25% BUY
    Expected profit: 4687.5

Blankets: 16.67% SELL
    Expected profit: 33333.33

Sleds: 4.17% SELL
    Expected profit: 2083.33

Sculptures: 6.25% BUY
    Expected profit: 4687.5

PS6: 10.42% BUY
    Expected profit: 13020.83

Serum: 25.0% SELL
    Expected profit: 75000.0

Lamps: 2.08% BUY
    Expected profit: 520.83

Chocolate: 2.08% SELL
    Expected profit: 520.83

Total expected profit: 134374.98
Total %/capital used: 75.0% = 750000.0 seashells
